# YOLO v3i Model/Image-Size Comparison

Controlled comparison on `dataset_yolo_bbox_v3i_li_archaeological_object_merged`.

Experiments:

| experiment | model | imgsz |
|---|---|---:|
| v3i_yolov8n_img640 | `yolov8n.pt` | 640 |
| v3i_yolov8n_img1024 | `yolov8n.pt` | 1024 |
| v3i_yolo26n_img640 | `yolo26n.pt` | 640 |
| v3i_yolo26n_img1024 | `yolo26n.pt` | 1024 |

Shared config:

```bash
epochs=300
batch=16
seed=42
single_cls=True
close_mosaic=10
patience=100
```

The notebook trains all four runs, validates best weights, runs threshold/proposal sweeps, creates comparison tables, and archives outputs.


In [ ]:
# Kaggle setup
import sys
import subprocess
from pathlib import Path

try:
    import ultralytics  # noqa: F401
except Exception:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "ultralytics"], check=True)


In [ ]:
from pathlib import Path

KAGGLE_INPUT_ROOT = Path("/kaggle/input")
PREBUILT_DATASET_DIR = Path(
    "/kaggle/input/datasets/matanerdy/detection-dataset/dataset_yolo_bbox_v3i_li_archaeological_object_merged"
)
WORK_ROOT = Path("/kaggle/working")
WORK_DATASETS_DIR = WORK_ROOT / "datasets"
DATASET_FOLDER = "dataset_yolo_bbox_v3i_li_archaeological_object_merged"
DATASET_WORK_DIR = WORK_DATASETS_DIR / DATASET_FOLDER
DATASET_YAML = DATASET_WORK_DIR / "dataset.yaml"

RUN_PROJECT = WORK_ROOT / "runs" / "archaeological_object_detection_model_imgsz_comparison"
ANALYSIS_DIR = WORK_ROOT / "analysis" / "v3i_model_imgsz_comparison"

EPOCHS = 300
PATIENCE = 100
BATCH = 16
RANDOM_SEED = 42
SINGLE_CLS = True
CLOSE_MOSAIC = 10
FORCE_RETRAIN = False

EXPERIMENTS = [
    {"experiment": "v3i_yolov8n_img640", "model": "yolov8n.pt", "imgsz": 640},
    {"experiment": "v3i_yolov8n_img1024", "model": "yolov8n.pt", "imgsz": 1024},
    {"experiment": "v3i_yolo26n_img640", "model": "yolo26n.pt", "imgsz": 640},
    {"experiment": "v3i_yolo26n_img1024", "model": "yolo26n.pt", "imgsz": 1024},
]

WORK_DATASETS_DIR.mkdir(parents=True, exist_ok=True)
RUN_PROJECT.mkdir(parents=True, exist_ok=True)
ANALYSIS_DIR.mkdir(parents=True, exist_ok=True)


## Prepare Dataset

In [ ]:
import shutil
import zipfile

import pandas as pd
import yaml


def find_dataset_dir(folder_name: str) -> Path | None:
    if PREBUILT_DATASET_DIR.exists():
        return PREBUILT_DATASET_DIR
    for dataset_yaml in KAGGLE_INPUT_ROOT.rglob("dataset.yaml"):
        parent = dataset_yaml.parent
        if parent.name == folder_name and (parent / "images").exists() and (parent / "labels").exists():
            return parent
    for metadata in KAGGLE_INPUT_ROOT.rglob("metadata.csv"):
        parent = metadata.parent
        if parent.name == folder_name and (parent / "images").exists() and (parent / "labels").exists():
            return parent
    return None


def find_dataset_zip(folder_name: str) -> Path | None:
    candidates = sorted(KAGGLE_INPUT_ROOT.rglob(f"{folder_name}.zip"))
    return candidates[0] if candidates else None


source_dir = find_dataset_dir(DATASET_FOLDER)

if DATASET_WORK_DIR.exists():
    shutil.rmtree(DATASET_WORK_DIR)

if source_dir is not None:
    print("Copying prebuilt dataset from:", source_dir)
    shutil.copytree(source_dir, DATASET_WORK_DIR)
else:
    zip_path = find_dataset_zip(DATASET_FOLDER)
    if zip_path is None:
        raise FileNotFoundError(f"Attach a Kaggle input containing {DATASET_FOLDER}/ or {DATASET_FOLDER}.zip")
    print("Extracting dataset zip:", zip_path)
    unzip_root = WORK_DATASETS_DIR / "_unzipped_v3i_archaeological_object"
    if unzip_root.exists():
        shutil.rmtree(unzip_root)
    unzip_root.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(zip_path, "r") as zf:
        zf.extractall(unzip_root)
    extracted_candidates = [p.parent for p in unzip_root.rglob("dataset.yaml") if p.parent.name == DATASET_FOLDER]
    extracted = extracted_candidates[0] if extracted_candidates else next(unzip_root.rglob("dataset.yaml")).parent
    shutil.copytree(extracted, DATASET_WORK_DIR)

DATASET_YAML.write_text(
    yaml.safe_dump(
        {
            "path": str(DATASET_WORK_DIR.resolve()),
            "train": "images/train",
            "val": "images/val",
            "names": {0: "archaeological_object"},
        },
        sort_keys=False,
        allow_unicode=True,
    ),
    encoding="utf-8",
)

metadata = pd.read_csv(DATASET_WORK_DIR / "metadata.csv")
if "is_target_object" in metadata.columns:
    boxes = metadata[metadata["is_target_object"].astype(bool)].copy()
else:
    boxes = metadata[metadata["class_name"].notna()].copy()
images = metadata.drop_duplicates("image").copy()

print("Dataset work dir:", DATASET_WORK_DIR)
print("Dataset yaml:", DATASET_YAML)
print("Images:", len(images))
print("Positive images:", images["is_positive"].astype(bool).sum() if "is_positive" in images.columns else images["image"].isin(set(boxes["image"])).sum())
print("BBox:", len(boxes))
display(images.groupby("split").size().rename("images").reset_index())
if "source_class_name" in boxes.columns:
    display(boxes.groupby(["split", "source_class_name"]).size().rename("bbox").reset_index())


## Train And Validate Four Experiments

In [ ]:
from ultralytics import YOLO

run_records = []

for cfg in EXPERIMENTS:
    experiment = cfg["experiment"]
    model_name = cfg["model"]
    imgsz = cfg["imgsz"]
    run_dir = RUN_PROJECT / experiment
    best_weights = run_dir / "weights" / "best.pt"
    results_csv = run_dir / "results.csv"

    print("=" * 100)
    print("Experiment:", experiment)
    print("Model:", model_name, "imgsz:", imgsz)

    if FORCE_RETRAIN or not best_weights.exists() or not results_csv.exists():
        model = YOLO(model_name)
        result = model.train(
            data=str(DATASET_YAML),
            imgsz=imgsz,
            epochs=EPOCHS,
            patience=PATIENCE,
            batch=BATCH,
            seed=RANDOM_SEED,
            deterministic=True,
            single_cls=SINGLE_CLS,
            close_mosaic=CLOSE_MOSAIC,
            project=str(RUN_PROJECT),
            name=experiment,
            exist_ok=True,
            plots=True,
        )
        run_dir = Path(result.save_dir)
        best_weights = run_dir / "weights" / "best.pt"
        results_csv = run_dir / "results.csv"
    else:
        print("Using existing run:", run_dir)

    best_model = YOLO(str(best_weights))
    val_result = best_model.val(
        data=str(DATASET_YAML),
        imgsz=imgsz,
        batch=BATCH,
        single_cls=SINGLE_CLS,
        project=str(RUN_PROJECT),
        name=f"{experiment}_val_best",
        exist_ok=True,
        plots=True,
    )

    run_records.append(
        {
            "experiment": experiment,
            "model": model_name,
            "imgsz": imgsz,
            "run_dir": str(run_dir),
            "best_weights": str(best_weights),
            "results_csv": str(results_csv),
            "val_dir": str(val_result.save_dir),
        }
    )

runs_df = pd.DataFrame(run_records)
runs_df.to_csv(ANALYSIS_DIR / "runs_index.csv", index=False)
display(runs_df)
print("Saved:", ANALYSIS_DIR / "runs_index.csv")


## Best-Metric Comparison

In [ ]:
metric_rows = []

for record in run_records:
    results = pd.read_csv(record["results_csv"])
    results.columns = [c.strip() for c in results.columns]
    best_idx = results["metrics/mAP50(B)"].idxmax()
    best = results.loc[best_idx]
    best_map5095_idx = results["metrics/mAP50-95(B)"].idxmax()
    best_map5095 = results.loc[best_map5095_idx]
    metric_rows.append(
        {
            "experiment": record["experiment"],
            "model": record["model"],
            "imgsz": record["imgsz"],
            "epochs_requested": EPOCHS,
            "epochs_completed": int(results["epoch"].max()),
            "best_epoch_mAP50": int(best["epoch"]),
            "Precision": float(best["metrics/precision(B)"]),
            "Recall": float(best["metrics/recall(B)"]),
            "mAP50": float(best["metrics/mAP50(B)"]),
            "mAP50-95": float(best["metrics/mAP50-95(B)"]),
            "best_epoch_mAP50-95": int(best_map5095["epoch"]),
            "best_mAP50-95": float(best_map5095["metrics/mAP50-95(B)"]),
            "run_dir": record["run_dir"],
            "best_weights": record["best_weights"],
        }
    )

metrics_summary = pd.DataFrame(metric_rows).sort_values("mAP50", ascending=False)
metrics_summary.to_csv(ANALYSIS_DIR / "model_imgsz_metrics_summary.csv", index=False)
display(metrics_summary)
print("Saved:", ANALYSIS_DIR / "model_imgsz_metrics_summary.csv")


## Threshold / Proposal Sweep Helpers

In [ ]:
import numpy as np
from PIL import Image

THRESHOLDS = [0.50, 0.25, 0.10, 0.05, 0.03, 0.01, 0.005, 0.003, 0.001]
MATCH_IOU = 0.50
COVERAGE_IOU = 0.30
ANALYSIS_CONF = min(THRESHOLDS)
WORKING_CONF = 0.25


def resolve_image_path(row: pd.Series) -> Path:
    split = str(row["split"])
    image_name = str(row.get("image_name") or Path(str(row["image"])).name)
    candidates = [
        DATASET_WORK_DIR / "images" / split / image_name,
        Path(str(row["image"])),
        DATASET_WORK_DIR / "images" / split / Path(str(row["image"])).name,
    ]
    for candidate in candidates:
        if candidate.exists():
            return candidate.resolve()
    return candidates[0].resolve()


def resolve_label_path(row: pd.Series) -> Path:
    split = str(row["split"])
    label_name = str(row.get("label_name") or Path(str(row.get("label", ""))).name or (Path(str(row.get("image_name") or row["image"])).stem + ".txt"))
    candidates = [
        DATASET_WORK_DIR / "labels" / split / label_name,
        Path(str(row.get("label", ""))),
        DATASET_WORK_DIR / "labels" / split / Path(str(row.get("label", label_name))).name,
        DATASET_WORK_DIR / "labels" / split / (Path(str(row.get("image_name") or row["image"])).stem + ".txt"),
    ]
    for candidate in candidates:
        if candidate.exists():
            return candidate.resolve()
    return candidates[0].resolve()


def parse_yolo_label_file(path: Path) -> list[tuple[int, float, float, float, float]]:
    if not path.exists():
        return []
    boxes = []
    for line in path.read_text().splitlines():
        parts = line.strip().split()
        if len(parts) != 5:
            continue
        cls_id = int(float(parts[0]))
        xc, yc, bw, bh = map(float, parts[1:])
        boxes.append((cls_id, xc, yc, bw, bh))
    return boxes


def box_iou(a: np.ndarray, b: np.ndarray) -> np.ndarray:
    if len(a) == 0 or len(b) == 0:
        return np.zeros((len(a), len(b)), dtype=float)
    x1 = np.maximum(a[:, None, 0], b[None, :, 0])
    y1 = np.maximum(a[:, None, 1], b[None, :, 1])
    x2 = np.minimum(a[:, None, 2], b[None, :, 2])
    y2 = np.minimum(a[:, None, 3], b[None, :, 3])
    inter = np.maximum(0, x2 - x1) * np.maximum(0, y2 - y1)
    area_a = np.maximum(0, a[:, 2] - a[:, 0]) * np.maximum(0, a[:, 3] - a[:, 1])
    area_b = np.maximum(0, b[:, 2] - b[:, 0]) * np.maximum(0, b[:, 3] - b[:, 1])
    union = area_a[:, None] + area_b[None, :] - inter
    return np.divide(inter, union, out=np.zeros_like(inter), where=union > 0)


def load_val_gt(metadata: pd.DataFrame) -> tuple[pd.DataFrame, list[Path]]:
    val = metadata[metadata["split"].astype(str).str.lower().eq("val")].copy()
    if "source_class_name" not in val.columns:
        val["source_class_name"] = val["class_name"]
    if "source_id" not in val.columns:
        source_cols = [col for col in ["region", "modality", "raster_file"] if col in val.columns]
        val["source_id"] = val[source_cols].astype(str).agg("|".join, axis=1)
    val["image_path"] = val.apply(resolve_image_path, axis=1)
    val["label_path"] = val.apply(resolve_label_path, axis=1)
    val["image_key"] = val["image_path"].map(lambda p: str(Path(p).resolve()))
    images_df = val.drop_duplicates("image_key").copy()
    image_paths = [Path(p).resolve() for p in images_df["image_path"]]
    rows = []
    gt_id = 0
    for _, image_row in images_df.iterrows():
        image_key = str(Path(image_row["image_path"]).resolve())
        label_path = Path(image_row["label_path"]).resolve()
        image_group = val[val["image_key"].eq(image_key)].copy()
        source_objects = image_group[image_group.get("is_target_object", image_group["class_name"].notna()).astype(bool)].reset_index(drop=True)
        source_classes = source_objects["source_class_name"].dropna().astype(str).tolist()
        with Image.open(image_key) as image:
            width, height = image.size
        label_boxes = parse_yolo_label_file(label_path)
        for local_idx, (cls_id, xc, yc, bw, bh) in enumerate(label_boxes):
            x1 = (xc - bw / 2) * width
            y1 = (yc - bh / 2) * height
            x2 = (xc + bw / 2) * width
            y2 = (yc + bh / 2) * height
            source_row = source_objects.iloc[local_idx] if local_idx < len(source_objects) else image_row
            source_class_name = source_classes[local_idx] if local_idx < len(source_classes) else "archaeological_object"
            rows.append(
                {
                    "gt_id": gt_id,
                    "image_key": image_key,
                    "label_path": str(label_path),
                    "region": source_row.get("region", image_row.get("region", "")),
                    "source_id": source_row.get("source_id", image_row.get("source_id", "")),
                    "raster_file": source_row.get("raster_file", image_row.get("raster_file", "")),
                    "source_class_name": source_class_name,
                    "class_name": "archaeological_object",
                    "objects_in_tile": len(label_boxes),
                    "bbox_width_px": (x2 - x1),
                    "bbox_height_px": (y2 - y1),
                    "bbox_area_px": (x2 - x1) * (y2 - y1),
                    "bbox_xyxy": (x1, y1, x2, y2),
                }
            )
            gt_id += 1
    return pd.DataFrame(rows), image_paths


def predict_dataframe(model: YOLO, image_paths: list[Path], imgsz: int, conf: float) -> pd.DataFrame:
    results = model.predict(
        source=[str(path) for path in image_paths],
        imgsz=imgsz,
        conf=conf,
        iou=0.50,
        verbose=False,
        save=False,
        stream=False,
    )
    rows = []
    for result_idx, result in enumerate(results):
        image_key = str(image_paths[result_idx].resolve())
        if result.boxes is None or len(result.boxes) == 0:
            continue
        xyxy = result.boxes.xyxy.cpu().numpy()
        confs = result.boxes.conf.cpu().numpy()
        for pred_idx, (box, score) in enumerate(zip(xyxy, confs)):
            rows.append({"image_key": image_key, "pred_idx": pred_idx, "x1": float(box[0]), "y1": float(box[1]), "x2": float(box[2]), "y2": float(box[3]), "confidence": float(score)})
    return pd.DataFrame(rows, columns=["image_key", "pred_idx", "x1", "y1", "x2", "y2", "confidence"])


def match_predictions(gt: pd.DataFrame, predictions: pd.DataFrame, match_iou: float, best_predictions: pd.DataFrame | None = None) -> tuple[pd.DataFrame, int]:
    audit = gt.copy()
    audit["is_found"] = False
    audit["matched_prediction_iou"] = 0.0
    audit["matched_prediction_confidence"] = np.nan
    audit["best_prediction_iou"] = 0.0
    audit["best_prediction_confidence"] = np.nan
    pred_by_image = {key: group.sort_values("confidence", ascending=False).reset_index(drop=True) for key, group in predictions.groupby("image_key")} if not predictions.empty else {}
    best_source = predictions if best_predictions is None else best_predictions
    best_by_image = {key: group.sort_values("confidence", ascending=False).reset_index(drop=True) for key, group in best_source.groupby("image_key")} if not best_source.empty else {}
    matched_prediction_keys: set[tuple[str, int]] = set()
    for image_key, gt_group in audit.groupby("image_key"):
        gt_indices = list(gt_group.index)
        gt_boxes = np.array(gt_group["bbox_xyxy"].tolist(), dtype=float)
        pred_group = pred_by_image.get(image_key, pd.DataFrame()).copy()
        best_group = best_by_image.get(image_key, pd.DataFrame()).copy()
        pred_boxes = pred_group[["x1", "y1", "x2", "y2"]].to_numpy(dtype=float) if not pred_group.empty else np.empty((0, 4))
        best_boxes = best_group[["x1", "y1", "x2", "y2"]].to_numpy(dtype=float) if not best_group.empty else np.empty((0, 4))
        ious = box_iou(pred_boxes, gt_boxes)
        best_ious = box_iou(best_boxes, gt_boxes)
        if len(best_boxes):
            for local_gt_idx, global_gt_idx in enumerate(gt_indices):
                best_pred_idx = int(np.argmax(best_ious[:, local_gt_idx]))
                audit.loc[global_gt_idx, "best_prediction_iou"] = float(best_ious[best_pred_idx, local_gt_idx])
                audit.loc[global_gt_idx, "best_prediction_confidence"] = float(best_group.iloc[best_pred_idx]["confidence"])
        candidates = []
        for pred_idx in range(len(pred_boxes)):
            for local_gt_idx, global_gt_idx in enumerate(gt_indices):
                iou_value = float(ious[pred_idx, local_gt_idx])
                if iou_value >= match_iou:
                    candidates.append((iou_value, float(pred_group.iloc[pred_idx]["confidence"]), pred_idx, global_gt_idx))
        matched_pred = set()
        matched_gt = set()
        for iou_value, confidence, pred_idx, global_gt_idx in sorted(candidates, reverse=True):
            if pred_idx in matched_pred or global_gt_idx in matched_gt:
                continue
            matched_pred.add(pred_idx)
            matched_gt.add(global_gt_idx)
            matched_prediction_keys.add((image_key, pred_idx))
            audit.loc[global_gt_idx, "is_found"] = True
            audit.loc[global_gt_idx, "matched_prediction_iou"] = iou_value
            audit.loc[global_gt_idx, "matched_prediction_confidence"] = confidence
    fp_count = max(0, len(predictions) - len(matched_prediction_keys))
    return audit, fp_count


def add_fn_types(audit: pd.DataFrame, conf_threshold: float) -> pd.DataFrame:
    audit = audit.copy()
    best_iou = pd.to_numeric(audit["best_prediction_iou"], errors="coerce").fillna(0.0)
    best_conf = pd.to_numeric(audit["best_prediction_confidence"], errors="coerce").fillna(0.0)
    audit["fn_type"] = "found"
    missed = ~audit["is_found"].astype(bool)
    metric_miss = missed & (best_iou >= MATCH_IOU) & (best_conf < conf_threshold)
    near_miss = missed & (~metric_miss) & (best_iou >= COVERAGE_IOU)
    hard_miss = missed & (~metric_miss) & (~near_miss)
    audit.loc[metric_miss, "fn_type"] = "metric_miss"
    audit.loc[near_miss, "fn_type"] = "near_miss"
    audit.loc[hard_miss, "fn_type"] = "hard_miss"
    return audit


val_gt, val_image_paths = load_val_gt(metadata)
print("Val images:", len(val_image_paths))
print("Val GT objects:", len(val_gt))


## Run Threshold Sweeps For Each Best Model

In [ ]:
all_sweep_rows = []
all_regional_rows = []

for record in run_records:
    experiment = record["experiment"]
    imgsz = int(record["imgsz"])
    print("=" * 100)
    print("Threshold sweep:", experiment)
    model = YOLO(record["best_weights"])
    predictions = predict_dataframe(model, val_image_paths, imgsz=imgsz, conf=ANALYSIS_CONF)
    predictions.to_csv(ANALYSIS_DIR / f"{experiment}_predictions_all_conf.csv", index=False)
    print("Predictions at analysis conf:", len(predictions))

    audits_by_threshold = {}
    for threshold in THRESHOLDS:
        preds = predictions[predictions["confidence"].ge(threshold)].copy()
        audit_t, fp = match_predictions(val_gt, preds, MATCH_IOU, best_predictions=predictions)
        audit_t = add_fn_types(audit_t, threshold)
        coverage_audit, _ = match_predictions(val_gt, preds, MATCH_IOU, best_predictions=preds)
        audits_by_threshold[threshold] = audit_t
        tp = int(audit_t["is_found"].sum())
        fn = int((~audit_t["is_found"]).sum())
        precision = tp / (tp + fp) if (tp + fp) else 0.0
        recall = tp / (tp + fn) if (tp + fn) else 0.0
        f1 = 2 * precision * recall / (precision + recall) if (precision + recall) else 0.0
        covered = int(pd.to_numeric(coverage_audit["best_prediction_iou"], errors="coerce").ge(COVERAGE_IOU).sum())
        all_sweep_rows.append(
            {
                "experiment": experiment,
                "model": record["model"],
                "imgsz": imgsz,
                "conf": threshold,
                "TP": tp,
                "FP": int(fp),
                "FN": fn,
                "Precision": precision,
                "Recall": recall,
                "F1": f1,
                "covered_gt_iou_0.30": covered,
                "total_gt": len(val_gt),
                "coverage_rate_iou_0.30": covered / len(val_gt) if len(val_gt) else 0.0,
                "predictions": int(len(preds)),
            }
        )

    working_audit = audits_by_threshold[WORKING_CONF].copy()
    working_audit.to_csv(ANALYSIS_DIR / f"{experiment}_gt_object_audit_conf_025.csv", index=False)
    for region, group in working_audit.groupby("region", dropna=False):
        gt_count = int(len(group))
        tp = int(group["is_found"].sum())
        fn_group = group[~group["is_found"]].copy()
        all_regional_rows.append(
            {
                "experiment": experiment,
                "model": record["model"],
                "imgsz": imgsz,
                "region": region,
                "GT": gt_count,
                "TP": tp,
                "FN": int(len(fn_group)),
                "recall": tp / gt_count if gt_count else 0.0,
                "metric_miss": int(fn_group["fn_type"].eq("metric_miss").sum()),
                "near_miss": int(fn_group["fn_type"].eq("near_miss").sum()),
                "hard_miss": int(fn_group["fn_type"].eq("hard_miss").sum()),
                "bbox_area_median": float(pd.to_numeric(group["bbox_area_px"], errors="coerce").median()),
                "classes": "; ".join(sorted(group["source_class_name"].dropna().astype(str).unique())),
            }
        )

threshold_sweep_all = pd.DataFrame(all_sweep_rows)
regional_audit_all = pd.DataFrame(all_regional_rows)
threshold_sweep_all.to_csv(ANALYSIS_DIR / "threshold_sweep_all.csv", index=False)
regional_audit_all.to_csv(ANALYSIS_DIR / "regional_audit_conf_025_all.csv", index=False)

display(threshold_sweep_all)
display(regional_audit_all)
print("Saved:", ANALYSIS_DIR / "threshold_sweep_all.csv")
print("Saved:", ANALYSIS_DIR / "regional_audit_conf_025_all.csv")


## Comparison Tables

In [ ]:
# Main detector comparison by best mAP50.
detector_comparison = metrics_summary[
    ["experiment", "model", "imgsz", "epochs_completed", "best_epoch_mAP50", "Precision", "Recall", "mAP50", "mAP50-95", "best_mAP50-95"]
].copy().sort_values("mAP50", ascending=False)

detector_comparison.to_csv(ANALYSIS_DIR / "detector_comparison.csv", index=False)
display(detector_comparison)

# Proposal-oriented comparison at selected thresholds.
proposal_thresholds = [0.25, 0.10, 0.05, 0.01, 0.005, 0.003]
proposal_comparison = threshold_sweep_all[threshold_sweep_all["conf"].isin(proposal_thresholds)].copy()
proposal_comparison = proposal_comparison[
    [
        "experiment",
        "model",
        "imgsz",
        "conf",
        "TP",
        "FP",
        "FN",
        "Precision",
        "Recall",
        "F1",
        "coverage_rate_iou_0.30",
        "predictions",
    ]
].sort_values(["conf", "coverage_rate_iou_0.30", "Recall"], ascending=[False, False, False])
proposal_comparison.to_csv(ANALYSIS_DIR / "proposal_comparison_by_threshold.csv", index=False)
display(proposal_comparison)

# Best F1 and best coverage rows per experiment.
best_f1 = threshold_sweep_all.loc[threshold_sweep_all.groupby("experiment")["F1"].idxmax()].copy().sort_values("F1", ascending=False)
best_coverage = threshold_sweep_all.loc[threshold_sweep_all.groupby("experiment")["coverage_rate_iou_0.30"].idxmax()].copy().sort_values("coverage_rate_iou_0.30", ascending=False)
best_f1.to_csv(ANALYSIS_DIR / "best_f1_thresholds.csv", index=False)
best_coverage.to_csv(ANALYSIS_DIR / "best_coverage_thresholds.csv", index=False)
display(best_f1)
display(best_coverage)


## Visual Artifacts

In [ ]:
from IPython.display import Image, display

for record in run_records:
    print("=" * 100)
    print(record["experiment"])
    run_dir = Path(record["run_dir"])
    for artifact in [
        run_dir / "results.png",
        run_dir / "confusion_matrix.png",
        run_dir / "BoxPR_curve.png",
        run_dir / "BoxF1_curve.png",
        run_dir / "val_batch0_labels.jpg",
        run_dir / "val_batch0_pred.jpg",
    ]:
        if artifact.exists():
            print(artifact)
            display(Image(filename=str(artifact)))
        else:
            print("Missing:", artifact)


## Write Report

In [ ]:
def md_table(df: pd.DataFrame) -> str:
    cols = [str(c) for c in df.columns]
    lines = ["| " + " | ".join(cols) + " |", "| " + " | ".join(["---"] * len(cols)) + " |"]
    for _, row in df.iterrows():
        lines.append("| " + " | ".join("" if pd.isna(v) else str(v) for v in row.tolist()) + " |")
    return "\n".join(lines)

report = [
    "# v3i Model/Image-Size Comparison Report",
    "",
    "## Dataset",
    "",
    f"- dataset: `{DATASET_FOLDER}`",
    f"- images: `{len(images)}`",
    f"- positive images: `{int(images['is_positive'].astype(bool).sum()) if 'is_positive' in images.columns else 'see metadata'}`",
    f"- bbox: `{len(boxes)}`",
    "",
    "## Shared Config",
    "",
    f"- epochs: `{EPOCHS}`",
    f"- patience: `{PATIENCE}`",
    f"- batch: `{BATCH}`",
    f"- seed: `{RANDOM_SEED}`",
    f"- single_cls: `{SINGLE_CLS}`",
    f"- close_mosaic: `{CLOSE_MOSAIC}`",
    "",
    "## Detector Comparison",
    "",
    md_table(detector_comparison),
    "",
    "## Proposal Comparison By Threshold",
    "",
    md_table(proposal_comparison),
    "",
    "## Best F1 Threshold Per Experiment",
    "",
    md_table(best_f1),
    "",
    "## Best Coverage Threshold Per Experiment",
    "",
    md_table(best_coverage),
    "",
    "## Regional Audit at conf=0.25",
    "",
    md_table(regional_audit_all.sort_values(["experiment", "recall", "GT"], ascending=[True, True, False])),
    "",
    "## Run Index",
    "",
    md_table(runs_df),
    "",
]

report_path = ANALYSIS_DIR / "v3i_model_imgsz_comparison_report.md"
report_path.write_text("\n".join(report), encoding="utf-8")
print("Saved:", report_path)
print("\n".join(report))


## Archive Outputs

In [ ]:
import zipfile
from datetime import datetime

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
archive_path = WORK_ROOT / f"yolo_v3i_model_imgsz_comparison_{timestamp}.zip"
roots_to_archive = [RUN_PROJECT, ANALYSIS_DIR]

with zipfile.ZipFile(archive_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    for root in roots_to_archive:
        if root.exists():
            for file in root.rglob("*"):
                if file.is_file():
                    zf.write(file, arcname=file.relative_to(WORK_ROOT))

print("Archive:", archive_path)
print("Archive size MB:", round(archive_path.stat().st_size / (1024 * 1024), 2))
